# Chat26 — ONE-CLICK C1 NONOFFICIAL EPOCH-1 ACCEPTANCE (MV00 + MV01)

**User-requested streamlined execution.** This notebook runs its complete pipeline in one Kaggle **Save Version → Save & Run All**, with no separate manual preparation cells. It performs exact release checkout, Chat25 frozen R0 20-pin/16-path restoration, six-input SHA resolution, two-arm plan verification and native dual-T4 epoch-1 acceptance. Every substantive scientific/data/security guard remains active.

**STRICT BOUNDARY:** User-enabled **nonofficial** acceptance for *one fresh epoch per model*. `official_training_authorized=False`, `b_test_access=NONE`, no official 100-epoch training. Never use this notebook as official training authorization.

**Kaggle requirements:** Fresh notebook session; accelerator **T4 ×2**; Internet enabled for fixed Git checkout; attach previously verified historical datasets and `nithoukhan/detection-project` containing `current_chat25_artifacts/CHAT25_C1_R0_STALE_PATH_SAFE_RESTAGE_CORRECTED.py`. Also attach exact Split B image dataset already referenced by runtime YAML.

**During startup:** Both models must scan their 14,227 training images. This can take many minutes and is not an epoch-progress indicator. The final cell displays status of each log at approximately 1-minute intervals. **Do not interrupt, manually restart or rerun cells in a used session.** The notebook captures logs into an incident ZIP if the dual supervisor exits unsuccessfully (when its Python process remains alive long enough to do so).

**Version provenance:** Original release `fa6a7675daae385a04c7b77e69a9de29f9e1b2f3`; this notebook is a separate orchestration artifact on `research/chat26-c1-acceptance-notebook`. Original frozen training modules and original Chat25 restager are unchanged.

**After success:** Preserve each epoch-1 original G4 archive, output index, logs, and SHA sidecar outside Kaggle; independent external verification and Version2 native epoch-2 resume remain separate mandatory later milestones.


In [ ]:
from pathlib import Path
import hashlib, json, os, io, runpy, shutil, subprocess, sys, zipfile
SHA="fa6a7675daae385a04c7b77e69a9de29f9e1b2f3"
BASE="b783eb3a470ab010785a8a24a90a666e9a3f2ed6"
PORTABLE_SHA="3f0a386ace43453921ddd84b0e5f1aef5dacb72493d11fd5fbadcfad3c974e3c"
PHASE="FRESH_EPOCH1"       # CHANGE TO "RESUME_EPOCH2" IN NEW KAGGLE VERSION
EXECUTE_ACCEPTANCE=True    # USER-REQUESTED NONOFFICIAL FRESH EPOCH-1 RUN ONLY
RID={"MV00":"C1ACCEPT_MV00_20261021","MV01":"C1ACCEPT_MV01_20261021"}
ROOT=Path("/kaggle/input")
REPO=Path("/kaggle/working/c1_official_fixed_checkout")
PLAN=Path("/kaggle/working/CHAT26_C1_PLANS_"+PHASE)
OUT=Path("/kaggle/working/CHAT26_C1_OUTPUT_"+PHASE)
LOG=Path("/kaggle/working/CHAT26_C1_LOGS_"+PHASE)
assert PHASE in ("FRESH_EPOCH1","RESUME_EPOCH2") and type(EXECUTE_ACCEPTANCE) is bool
assert ROOT.is_dir() and not any(p.exists() or p.is_symlink() for p in (REPO,PLAN,OUT,LOG))
# Fast fail on accelerator mismatch before expensive R0/data verification.
gpu_list=subprocess.run(["nvidia-smi","-L"],capture_output=True,text=True)
assert gpu_list.returncode==0, "T4_X2_GPU_NOT_ENABLED"
gpu_lines=[line for line in gpu_list.stdout.splitlines() if line.startswith("GPU ")]
assert len(gpu_lines)==2 and all("T4" in line.upper() for line in gpu_lines), "PHYSICAL_T4_X2_REQUIRED"
print("ONE_CLICK_T4_X2_PRECHECK_PASS",*gpu_lines,sep="\\n",flush=True)
def digest(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for b in iter(lambda:f.read(1048576),b""):h.update(b)
    return h.hexdigest()
print("NONOFFICIAL_C1_ACCEPTANCE_ONLY",PHASE,"OFFICIAL_AUTHORIZATION_FALSE; B_TEST_NONE")


In [ ]:
subprocess.run(["git","clone","--no-checkout","https://github.com/nithoukhan1/ResEMA.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--detach",SHA],check=True)
assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()==SHA
assert REPO.is_dir() and not REPO.is_symlink(), "R0_SECURITY_FORBIDS_SYMLINK_CHECKOUT"
assert not any(x.is_symlink() for x in (REPO, REPO.parent)), "R0_SYMLINK_ANCESTOR_FORBIDDEN"
REL=REPO/"research/r9_training"; C1=REL/"official_candidate"; CODE=REL/"code"
sys.path[:0]=[str(C1),str(CODE)]
from R9_OFFICIAL_RELEASE_IDENTITY_R8 import validate_checkout
assert validate_checkout(REPO,SHA)["status"]=="R8_EXACT_CHECKOUT_SHA_AND_ADDITIVE_DIFF_VERIFIED_NOT_REMOTE_PROOF"
print("EXACT_C1_RELEASE_PASS",SHA)


In [ ]:
# Execute exact tested corrected R0 restager from the migration source, never an approximation.
NAME="CHAT25_C1_R0_STALE_PATH_SAFE_RESTAGE_CORRECTED.py"
def safe_member(s):
    from pathlib import PurePosixPath
    p=PurePosixPath(s)
    return not p.is_absolute() and ".." not in p.parts and "\\" not in s
def nested_scan(blob,depth=0):
    if depth>3:return []
    out=[]
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for m in z.infolist():
            if m.is_dir() or not safe_member(m.filename):continue
            if m.filename==NAME or m.filename.endswith("/"+NAME):
                assert m.file_size<2_000_000
                out.append(z.read(m))
            elif m.filename.lower().endswith(".zip") and m.file_size<120_000_000:
                out+=nested_scan(z.read(m),depth+1)
    return out
parts=[p.read_bytes() for p in ROOT.rglob(NAME) if p.is_file() and not p.is_symlink()]
for z in ROOT.rglob("*"):
    if z.is_file() and "DETECTION_PROJECT_MASTER_MIGRATION_CHAT1_TO_CHAT25_FOR_CHAT26_2026-10-21_PORTABLE" in z.name and z.suffix in (".zip",".zipblob"):
        assert digest(z)==PORTABLE_SHA, "PORTABLE_MIGRATION_SHA_MISMATCH"
        parts+=nested_scan(z.read_bytes())
if not parts:
    print("KAGGLE_MOUNT_ROOTS:", [str(x) for x in ROOT.iterdir()])
    print("MIGRATION_CANDIDATES:", [str(x) for x in ROOT.rglob("*") if "CHAT1_TO_CHAT25" in x.name][:30])
    raise RuntimeError("MISSING_CHAT25_CORRECTED_R0_SCRIPT: add Chat26 portable ZIP as a Kaggle dataset, or add the extracted current_chat25_artifacts/CHAT25_C1_R0_STALE_PATH_SAFE_RESTAGE_CORRECTED.py")
assert len({hashlib.sha256(b).hexdigest() for b in parts})==1,"CONFLICTING_RESTAGERS"
s=Path("/kaggle/working")/NAME;s.write_bytes(parts[0])
print("CORRECTED_RESTAGER_HASH",digest(s))
# Validate the exact original Chat25 checkout prerequisites BEFORE restaging.
# Frozen R0 safe_file() prohibits symlinks in its ancestry. Never create aliases.
assert digest(s)=="88cd4a88c373f451e54ba64736e961e735757d94b3779feea55a526868c45f4f", "UNEXPECTED_CORRECTED_RESTAGER_BYTES"
expected_checkout=Path("/kaggle/working/c1_official_fixed_checkout")
expected_config=Path("/kaggle/input/datasets/nithoukhan2/detect-1/R7D_RESTORE_CONTEXT_SUPPORT_R1/inputs/RECOVERY_CONFIG.json")
expected_sci=expected_checkout/"research/r9_training/frozen_source/ultralytics/models/yolo/detect/d4e_r3_read_only_identity_preflight.py"
assert REPO==expected_checkout and REPO.is_dir() and not REPO.is_symlink(), "MUST_USE_REAL_CHAT25_CHECKOUT_DIRECTORY"
assert expected_config.is_file() and not expected_config.is_symlink(), "ORIGINAL_R7D_CONFIG_MISSING"
assert expected_sci.is_file() and not expected_sci.is_symlink(), "ORIGINAL_R0_SOURCE_MISSING"
assert not any(p.is_symlink() for p in (expected_checkout, expected_checkout.parent, expected_sci, expected_sci.parent)), "R0_SYMLINK_ANCESTOR_FORBIDDEN"
assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()==SHA
print("CHAT25_REAL_CHECKOUT_PRECONDITIONS_PASS_NO_SYMLINKS")
runpy.run_path(str(s),run_name="__main__")
print("CORRECTED_R0_RESTAGER_EXECUTED; R0 WILL_BE_REVALIDATED_BY_C1_PLAN_FACTORY")


In [ ]:
# Immutable SHA-based map. Only original mounted dataset files may be used as C1 inputs.
PINS={
 "runtime_yaml":("f2cc6e31f5f1be152f4dd7603a397e142963923f1d35de613eb7c61b168bf0c0",(".yaml",".yml")),
 "assignment":("87aa5e9089c6282df14ae32917946d70d9f56eaa9a5d63443985877ab9dd1ef4",(".csv",".json")),
 "visibility":("9de034fa7df668d17effa843674c097341af9dd6461963f9e2c454c5f358b289",(".json",)),
 "pretrained_best":("620594d3560310b5d24243e1b321ce4463b0795b295891c47f6ec177e5272ac1",(".pt",)),
 "pretrained_archive":("4e3a811be5b8126e788ca4b7854e2df2b070a08e2ab8f0c6bbde8b6e1abef396",(".bin",".zip",".zipblob"))
}
MOUNTED=("d4e-r12-accepted-assets","d4e-r12-raw-archives-v1","detect-1","detection","d4f-r7d-legacy-exact-source-evidence-r1")
files=[]
for dirpath,dirs,fnames in os.walk(ROOT):
    dirs[:]=[d for d in dirs if not d.startswith(".")]
    for f in fnames:
        p=Path(dirpath)/f
        if any(m in str(p) for m in MOUNTED) and p.is_file() and not p.is_symlink():files.append(p)
INPUT={}
for key,(pin,suffixes) in PINS.items():
    found=[p for p in files if p.name.lower().endswith(suffixes) and digest(p)==pin]
    assert found, "MISSING_EXACT_PINNED_ASSET_"+key
    INPUT[key]=str(sorted(found,key=str)[0])
configs=[p for p in files if p.name=="RECOVERY_CONFIG.json" and "R7D_RESTORE_CONTEXT_SUPPORT_R1/inputs" in str(p)]
assert len(configs)==1,"R7D_ORIGINAL_RECOVERY_CONFIG_MOUNT_REQUIRED"
INPUT["config_path"]=str(configs[0])
print("SIX_INPUT_MAP_EXACT_BYTES_PASS",json.dumps(INPUT,indent=2))


In [ ]:
# Version2 must mount independently archived epoch1 bytes and external attestations.
from R9_OFFSESSION_TRANSPORT_R1 import inspected_archive
def resume_triplet(arm):
    options=[]
    for blob in ROOT.rglob("*_epoch001_G4_LOCAL_EXPORT.zip.zipblob"):
        if not blob.is_file() or arm not in str(blob):continue
        side=Path(str(blob)+".sha256")
        if not side.is_file() or side.read_text().split()!=[digest(blob),blob.name]:continue
        chk=inspected_archive(blob,sidecar=side)
        if (chk["arm"],chk["completed_epoch"],chk["terminal"])!=(arm,0,False):continue
        for a in blob.parent.rglob("*.json"):
            try:v=json.loads(a.read_text())
            except (OSError,ValueError):continue
            if (v.get("schema")=="R9_OFFSITE_CHECK_NOT_NATIVE_RESUME_ACCEPTANCE"
                and v.get("arm")==arm and v.get("completed_epoch")==0
                and v.get("zip_sha256")==digest(blob)):
                options.append((str(blob),str(side),str(a)))
    assert len(options)==1,"EXACT_ONE_VERIFIED_OFFSESSION_EPOCH1_TRIPLET_REQUIRED_"+arm
    return options[0]
RESUME={arm:resume_triplet(arm) for arm in RID} if PHASE=="RESUME_EPOCH2" else {}
print("RESUME_EVIDENCE_MOUNT_GATE",("BOTH_ARMS_PASS" if RESUME else "FRESH_NO_RESUME"))


In [ ]:
from R9_PRODUCTION_PLAN_FACTORY_C1 import create
from R9_PRODUCTION_APPROVAL_GATE_C1 import validate_acceptance,ACCEPTANCE_SCHEMA,ACCEPTANCE_PURPOSE
epoch=1 if PHASE=="FRESH_EPOCH1" else 2
mode="FRESH_FROM_EARLY" if epoch==1 else "RESUME_FROM_OUR_OWN_RUN"
arms={}
for a in RID:
    z=(None,None,None) if epoch==1 else RESUME[a]
    arms[a]={"mode":mode,"start_epoch_one_based":epoch,"target_epoch_one_based":epoch,
             "resume_archive":z[0],"resume_sidecar":z[1],"resume_external_attestation":z[2]}
request={"run_ids":RID,"arms":arms,"monotonic_session_seconds":32400,
         "minimum_epoch_seconds":1800,"reserve_export_seconds":3600}
result=create(INPUT,request,checkout=str(REPO),release_sha=SHA,output_dir=str(PLAN))
assert result["status"]=="TWO_ARM_PINNED_PLANS_CREATED_NO_GPU"
specs={a:json.loads((PLAN/(a+".json")).read_text()) for a in RID}
print("NONOFFICIAL_TWO_ARM_PLAN_PASS_NO_GPU",json.dumps(result,indent=2))


In [ ]:
# Last cell — launch exactly one governed native dual-worker execution.
# Do not run this cell separately after a Save & Run All version has begun.
assert PHASE=="FRESH_EPOCH1" and EXECUTE_ACCEPTANCE is True
import time
from datetime import datetime, timezone
import torch
assert torch.cuda.is_available() and torch.cuda.device_count()==2, "CUDA_T4_X2_REQUIRED"
assert all("T4" in torch.cuda.get_device_name(i).upper() for i in (0,1))
assert not OUT.exists() and not LOG.exists(), "FRESH_OUTPUT_AND_LOG_ROOTS_REQUIRED"
assert all(not Path(specs[arm]["working_root"]).exists() for arm in RID)
assert all(specs[arm]["mode"]=="FRESH_FROM_EARLY"
           and specs[arm]["start_epoch_one_based"]==1
           and specs[arm]["target_epoch_one_based"]==1
           and specs[arm]["resume_archive"] is None for arm in RID)

consent={"schema":ACCEPTANCE_SCHEMA,"purpose":ACCEPTANCE_PURPOSE,
         "release_commit":SHA,"frozen_science_commit":BASE,"run_ids":RID,
         "official_training_authorized":False,"b_test_access":"NONE",
         "max_completed_epoch_one_based":2}
validate_acceptance(specs,consent,SHA)
permit=PLAN/"C1_USER_ENABLED_NONOFFICIAL_ACCEPTANCE.json"
with permit.open("x",encoding="utf-8") as f:
    json.dump(consent,f,sort_keys=True,indent=2);f.write("\n")
cmd=[sys.executable,"-u",str(C1/"R9_PRODUCTION_DUAL_SESSION_C1.py"),
     "--mv00",str(PLAN/"MV00.json"),"--mv01",str(PLAN/"MV01.json"),
     "--release-sha",SHA,"--authorization",str(permit),
     "--output-root",str(OUT),"--logs-root",str(LOG),
     "--execute-nonofficial-finalpath-acceptance"]
supervisor=Path("/kaggle/working/CHAT26_C1_ONECLICK_EPOCH1_SUPERVISOR.log")
incident=Path("/kaggle/working/CHAT26_C1_ONECLICK_EPOCH1_INCIDENT.zip")
assert not supervisor.exists() and not incident.exists()
def tail_status(path):
    if not path.is_file():
        return "not-created-yet"
    # Read only a bounded tail to avoid repeated full log processing.
    with path.open("rb") as f:
        f.seek(max(0,path.stat().st_size-8192))
        portion=f.read().decode("utf-8",errors="replace")
    lines=[x.strip() for x in portion.replace("\r","\n").splitlines() if x.strip()]
    return (lines[-1][:200] if lines else "empty")
def save_incident():
    files=[PLAN/"MV00.json",PLAN/"MV01.json",PLAN/"PLAN_REVIEW.json",permit,supervisor]
    files += [LOG/(arm+"_native_stdout_stderr.log") for arm in RID]
    files += [OUT/"C1_OUTPUT_INDEX.json"]
    with zipfile.ZipFile(incident,"x",zipfile.ZIP_DEFLATED) as z:
        for p in files:
            if p.is_file() and not p.is_symlink():
                z.write(p,p.relative_to("/kaggle/working").as_posix())
    h=digest(incident)
    Path(str(incident)+".sha256").write_text(h+"  "+incident.name+"\n")
    print("FAILED_RUN_INCIDENT_ARCHIVE:",incident,"SHA256:",h,flush=True)

print("NONOFFICIAL_DUAL_T4_WORKERS_STARTING",flush=True)
print("INITIAL_LABEL_SCAN_CAN_TAKE_MANY_MINUTES; DO_NOT_INTERRUPT",flush=True)
with supervisor.open("x",encoding="utf-8") as sink:
    proc=subprocess.Popen(cmd,stdout=sink,stderr=subprocess.STDOUT,
                          stdin=subprocess.DEVNULL)
    try:
        while proc.poll() is None:
            print("HEARTBEAT_UTC:",datetime.now(timezone.utc).isoformat(),flush=True)
            for arm in RID:
                print(arm,"::",tail_status(LOG/(arm+"_native_stdout_stderr.log")),flush=True)
            time.sleep(60)
    except KeyboardInterrupt:
        print("INTERRUPTED_MONITOR: DO_NOT_RESTART_IN_THIS_SESSION",flush=True)
        # Retain the failed run's original logs; avoid running any second worker.
        if proc.poll() is None:
            proc.terminate()
            try: proc.wait(timeout=30)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
        save_incident()
        raise
    rc=proc.wait()
print("DUAL_SESSION_EXIT_CODE:",rc,flush=True)
if rc!=0:
    save_incident()
    print("DUAL_SUPERVISOR_TAIL:\n",tail_status(supervisor),flush=True)
    for arm in RID:
        print(arm,"FINAL_LOG_TAIL:",tail_status(LOG/(arm+"_native_stdout_stderr.log")),flush=True)
    raise RuntimeError("C1_EPOCH1_FAILED_KEEP_INCIDENT_AND_NATIVE_LOGS_NO_RETRY_THIS_SESSION")

idx_path=OUT/"C1_OUTPUT_INDEX.json"
if not idx_path.is_file():
    save_incident()
    raise RuntimeError("MISSING_C1_OUTPUT_INDEX")
idx=json.loads(idx_path.read_text())
if (idx.get("both_arms_completed") is not True or idx.get("failures")
    or set(idx.get("arms",{}))!=set(RID)):
    save_incident()
    raise RuntimeError("NONOFFICIAL_DUAL_ARM_COMPLETION_FAILED")
assert idx["run_class"]=="NONOFFICIAL_FINALPATH_ACCEPTANCE"
assert idx["offsite_persistence_verified"] is False
assert idx["b_test_access"]=="NONE"
assert idx["published_release_commit"]==SHA
assert idx["status"]=="BOTH_ARMS_LOCAL_OUTPUTS_VERIFIED_EXTERNAL_PENDING"
for arm,rec in idx["arms"].items():
    assert rec["run_id"]==RID[arm]
    assert rec["completed_epoch_zero_based"]==0 and rec["terminal"] is False
    blob=OUT/rec["zip_relative"]
    assert blob.is_file() and blob.name.endswith(".zipblob")
    assert digest(blob)==rec["zip_sha256"]
    side=Path(str(blob)+".sha256")
    with side.open("x",encoding="utf-8") as f:
        f.write(rec["zip_sha256"]+"  "+blob.name+"\n")
    shutil.copy2(LOG/(arm+"_native_stdout_stderr.log"),
                 OUT/(arm+"_native_stdout_stderr.log"))
    print("LOCAL_G4_EPOCH1_ARCHIVE_PASS",arm,blob.name,rec["zip_sha256"],flush=True)
shutil.copy2(supervisor,OUT/"CHAT26_C1_SUPERVISOR_STDOUT_STDERR.log")
print("BOTH_ARMS_LOCAL_G4_PASS_OFFSITE_UNVERIFIED_SAVE_KAGGLE_VERSION",flush=True)


## Outcome and migration boundary

On **successful** Save & Run All execution, inspect `CHAT26_C1_OUTPUT_FRESH_EPOCH1/C1_OUTPUT_INDEX.json` (both arms completed), two nonofficial epoch-1 G4 .zipblob files with freshly generated filename-correct .zipblob.sha256 sidecars, and two native logs plus supervisor log. **Local G4 is not offsite persistence proof.** Preserve and verify both archives externally, then separately test genuine epoch-2 native resume in a new Kaggle version. Do not convert this acceptance into an official 100-epoch claim or touch B-TEST.

If the dual supervisor returns a failure and notebook control continues, the notebook produces a `CHAT26_C1_ONECLICK_EPOCH1_INCIDENT.zip` with source plans and worker logs. If the entire Kaggle version is forcibly terminated, this Python failure-capture step may not execute; recover output from the saved version logs.
